In [ ]:
import torch
import torch.nn as nn
import torch.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset , DataLoader
from sklearn.datasets import  load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
device=torch.device("cuda"if torch.cuda.is_available() else "cpu")

In [ ]:
scaler=StandardScaler()

In [ ]:

data=load_breast_cancer()
data.data=scaler.fit_transform(data.data)
data.data=torch.from_numpy(data.data).float()
data.target=torch.from_numpy(data.target).float()


In [ ]:
X_train,X_test,y_train,y_test=train_test_split(data.data,data.target,test_size=0.2,random_state=42)
X_train

tensor([[-1.4480, -0.4560, -1.3667,  ...,  0.9196,  2.1472,  1.8594],
        [ 1.9775,  1.6942,  2.0896,  ...,  2.6752,  1.9369,  2.4635],
        [-1.4071, -1.2635, -1.3498,  ..., -0.9705,  0.6168,  0.0529],
        ...,
        [ 0.0462, -0.5747, -0.0687,  ..., -1.2376, -0.7163, -1.2605],
        [-0.0418,  0.0769, -0.0350,  ...,  1.0368,  0.4501,  1.1944],
        [-0.5531,  0.2863, -0.6075,  ..., -0.6136, -0.3345, -0.8404]])

In [ ]:
model=nn.Sequential(
    nn.Linear(30,32),
    nn.ReLU(),
    nn.Linear(32,16),
    nn.ReLU(),
    nn.Linear(16,1),
    nn.Sigmoid()
).to(device)

In [ ]:
train_dataset=TensorDataset(X_train,y_train)
test_dataset=TensorDataset(X_test,y_test)
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=True)

In [ ]:
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

In [ ]:
num_epoch=50
for epoch in range(num_epoch):
  model.train()
  running_loss=0.0
  for input_batch,label_batch in train_loader:
    input_batch=input_batch.to(device)
    label_batch=label_batch.to(device)
    optimizer.zero_grad()
    output=model(input_batch)
    loss=criterion(output,label_batch.unsqueeze(1).float())
    loss.backward()
    optimizer.step()
    running_loss+=loss.item()
  if (epoch+1)%10==0:
    print(f"Epoch {epoch+1}/{num_epoch} : loss = {running_loss/len(train_loader)}")


Epoch 10/50 : loss = 0.10704751759767532
Epoch 20/50 : loss = 0.05450178074340026
Epoch 30/50 : loss = 0.035891669367750484
Epoch 40/50 : loss = 0.026504475918288032
Epoch 50/50 : loss = 0.01648841070321699


In [ ]:
model.eval()
correct=0
total=0
with torch.no_grad():
  for inputs,labels in test_loader:
    inputs=inputs.to(device)
    labels=labels.to(device)
    outputs=model(inputs)
    prediction=(outputs>=0.5).float()
    correct+=(prediction.squeeze()==labels).sum().item()
    total+=labels.size(0)
print(f"accuracy = {correct*100/total}")

accuracy = 98.24561403508773
